<a href="https://colab.research.google.com/github/anirbanghoshsbi/.github.io/blob/master/work/err/cross_section_momentum/temp_work_file.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:

#----------Read ZIP Files Extract the CSV files and put them in a folder levelone---------
import os
import zipfile
import shutil
import pandas as pd
import numpy as np

source_folder = "/content"           # Folder containing ZIP files
levelone_folder = "/content/levelone"

os.makedirs(levelone_folder, exist_ok=True)

def extract_all_zips(source, destination):

    # First copy all zip files to levelone
    for file in os.listdir(source):
        if file.endswith(".zip"):
            shutil.copy(os.path.join(source, file), destination)

    # Now recursively extract inside levelone
    while True:
        zip_found = False

        for root, dirs, files in os.walk(destination):
            for file in files:
                if file.endswith(".zip"):
                    zip_found = True
                    zip_path = os.path.join(root, file)

                    with zipfile.ZipFile(zip_path, 'r') as z:
                        z.extractall(root)

                    os.remove(zip_path)
                    print(f"Extracted and removed: {file}")

        if not zip_found:
            break

    print("All ZIP files fully extracted into levelone.")


extract_all_zips(source_folder, levelone_folder)

In [5]:
#-----Connect my Coding Terminal to Google Drive------
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [6]:
#Import Database using duckdb
import duckdb

with duckdb.connect("/content/drive/MyDrive/All_equities_2020_onwards.duckdb") as con:
    df = con.execute("SELECT * FROM equities").fetchdf()

print(df.shape)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

(3610528, 13)


In [7]:
df.columns

Index(['SYMBOL', 'SERIES', 'OPEN', 'HIGH', 'LOW', 'CLOSE', 'LAST', 'PREVCLOSE',
       'TOTTRDQTY', 'TOTTRDVAL', 'TIMESTAMP', 'TOTALTRADES', 'ISIN'],
      dtype='object')

In [8]:
df.tail(5)

,SYMBOL,SERIES,OPEN,HIGH,LOW,CLOSE,LAST,PREVCLOSE,TOTTRDQTY,TOTTRDVAL,TIMESTAMP,TOTALTRADES,ISIN
3610523,ZOTA,EQ,1208.50,1220.00,1177.15,1201.50,1200.65,1195.10,61764,7.424303e+07,2025-07-25,7246,INE358U01012
3610524,ZUARI,EQ,217.56,217.56,212.10,213.57,213.50,218.14,109229,2.336998e+07,2025-07-25,2381,INE840M01016
3610525,ZUARIIND,EQ,268.90,272.00,264.15,265.10,265.20,270.00,32471,8.655200e+06,2025-07-25,1135,INE217A01012
3610526,ZYDUSLIFE,EQ,967.20,981.95,960.50,976.40,973.50,971.25,1188906,1.159735e+09,2025-07-25,39311,INE010B01027
3610527,ZYDUSWELL,EQ,2149.40,2157.40,2092.90,2102.60,2093.40,2138.10,44772,9.463579e+07,2025-07-25,8027,INE768C01010


In [9]:
# We are appending all the csv files into a single pandas dataframe
import pandas as pd
import glob
import os

# Path to folder containing CSV files
folder_path = "/content/levelone"

# Get all CSV files
csv_files = glob.glob(os.path.join(folder_path, "*.csv"))

# Read and append
df_list = []

for file in csv_files:
    temp_df = pd.read_csv(file)
    df_list.append(temp_df)

# Combine all into one dataframe
full_data_from_csv = pd.concat(df_list, ignore_index=True)

In [10]:
full_data_from_csv.columns

Index(['TradDt', 'BizDt', 'Sgmt', 'Src', 'FinInstrmTp', 'FinInstrmId', 'ISIN',
       'TckrSymb', 'SctySrs', 'XpryDt', 'FininstrmActlXpryDt', 'StrkPric',
       'OptnTp', 'FinInstrmNm', 'OpnPric', 'HghPric', 'LwPric', 'ClsPric',
       'LastPric', 'PrvsClsgPric', 'UndrlygPric', 'SttlmPric', 'OpnIntrst',
       'ChngInOpnIntrst', 'TtlTradgVol', 'TtlTrfVal', 'TtlNbOfTxsExctd',
       'SsnId', 'NewBrdLotQty', 'Rmks', 'Rsvd1', 'Rsvd2', 'Rsvd3', 'Rsvd4'],
      dtype='object')

In [11]:
df.columns

Index(['SYMBOL', 'SERIES', 'OPEN', 'HIGH', 'LOW', 'CLOSE', 'LAST', 'PREVCLOSE',
       'TOTTRDQTY', 'TOTTRDVAL', 'TIMESTAMP', 'TOTALTRADES', 'ISIN'],
      dtype='object')

In [12]:
# Remove unwanted columns
#df = df.loc[:, ~df.columns.str.contains("^Unnamed")]
#------#----------------#----------------#---------#

#We Do a Column Mapping So that the Pandas Dataframe and the Duckdb Dataframe has the same Column Names
# Column mapping
column_mapping = {
    "TckrSymb": "SYMBOL",
    "SctySrs": "SERIES",
    "OpnPric": "OPEN",
    "HghPric": "HIGH",
    "LwPric": "LOW",
    "ClsPric": "CLOSE",
    "LastPric": "LAST",
    "PrvsClsgPric": "PREVCLOSE",
    "TtlTradgVol": "TOTTRDQTY",
    "TtlTrfVal": "TOTTRDVAL",
    "BizDt": "TIMESTAMP",
    "TtlNbOfTxsExctd": "TOTALTRADES",
    "ISIN": "ISIN"
}

# Subset + rename
subset_full_data = full_data_from_csv[list(column_mapping.keys())].copy()
subset_full_data.rename(columns=column_mapping, inplace=True)



In [13]:
subset_full_data.tail(2)

,SYMBOL,SERIES,OPEN,HIGH,LOW,CLOSE,LAST,PREVCLOSE,TOTTRDQTY,TOTTRDVAL,TIMESTAMP,TOTALTRADES,ISIN
144077,ZYDUSLIFE,EQ,902.05,907.00,899.1,903.85,907.00,907.95,369018,3.334368e+08,2026-02-24,17701,INE010B01027
144078,ZYDUSWELL,EQ,406.65,406.65,395.0,399.00,398.75,406.65,69035,2.756971e+07,2026-02-24,3777,INE768C01028


In [14]:
# Concatenate vertically
combined_df = pd.concat(
    [df, subset_full_data],
    ignore_index=True
)

print(combined_df.shape)

(3754607, 13)


In [15]:
combined_df.shape

(3754607, 13)

In [19]:
# Save the Dataframe as DuckDB

# Name of database file you want to create
db_path = "/content/drive/MyDrive/All_equities_2020_onwards.duckdb"

# Connect (this creates the file if it doesn't exist)
with duckdb.connect(db_path) as con:
    con.execute("CREATE OR REPLACE TABLE equities AS SELECT * FROM combined_df")

print("DuckDB database created successfully.")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

DuckDB database created successfully.
